In [3]:
import sys
from pathlib import Path

project_root = Path.cwd().resolve().parent

src_path = project_root / 'src'
if str(src_path) not in sys.path:
    sys.path.append(str(src_path))
    
from config import *
print(f"Environment linked. Data dir: {DATA_DIR}")

Environment linked. Data dir: /home/itapit/github-repos/LucidRF/ml/data


In [4]:
from scipy.stats import kurtosis
import h5py
import numpy as np

In [5]:
class MachineA_Preprocessor:
    def __init__(self):
        self.f = h5py.File(RAW_DATA_FILE, 'r')
        self.splits = np.load(SPLIT_MAP_FILE)

    def process_and_save(self):
        # We only prepare train and validation sets now, test is saved for later
        self._process_subset('train')
        self._process_subset('val')

    def _process_subset(self, split_name):
        print(f"\n--- Processing {split_name.upper()} Set ---")
        indices = self.splits[split_name]
        
        # for now we only use a subset of data for speed
        limit = 1000
        if len(indices) > limit:
            print(f"sampling {limit} from {len(indices)} available samples.")
            # Sort first (for HDF5 speed), then take the first N
            indices = np.sort(indices)[:limit]
        else:
            indices = np.sort(indices)
        
        # Load Clean Data
        print("Loading Clean I/Q Data...")
        clean_iq = self.f['X'][indices]
        
        # Extract Features -> Label 0 (Clean)
        print("Extracting Features for CLEAN signals...")
        feats_clean = self._extract_features(clean_iq)
        labels_clean = np.zeros(len(feats_clean))
        
        # Generate Noise -> Label 1 (Noisy)
        print("Generating Barrage Noise & Extracting Features...")
        noisy_iq = self._add_barrage_noise(clean_iq)
        feats_noisy = self._extract_features(noisy_iq)
        labels_noisy = np.ones(len(feats_noisy))
        
        # Combine
        X = np.vstack([feats_clean, feats_noisy])
        Y = np.concatenate([labels_clean, labels_noisy])
        
        x_path = PROCESSED_DIR / f'machine_a_X_{split_name}.npy'
        y_path = PROCESSED_DIR / f'machine_a_Y_{split_name}.npy'
        
        np.save(x_path, X)
        np.save(y_path, Y)
        print(f"Saved: {x_path.name} (Shape: {X.shape})")

    def _add_barrage_noise(self, iq_samples, noise_power_db=5):
        # Barrage Interference (Broadband Gaussian Noise)
        n, l, _ = iq_samples.shape
        power = 10 ** (noise_power_db / 10.0)
        noise = (np.random.normal(0, np.sqrt(power/2), (n, l)) + 
                 1j * np.random.normal(0, np.sqrt(power/2), (n, l)))
        return iq_samples + np.stack([noise.real, noise.imag], axis=2)

    def _extract_features(self, iq_batch):
        # Calculates: Mean Power, Variance, PAPR, Kurtosis
        # Convert to complex numbers for easier math
        complex_data = iq_batch[:, :, 0] + 1j * iq_batch[:, :, 1]
        
        # Mean Power
        power = np.abs(complex_data)**2
        mean_power = np.mean(power, axis=1)
        
        # Variance
        variance = np.var(complex_data, axis=1)
        
        # PAPR (Peak-to-Average Power Ratio)
        peak = np.max(power, axis=1)
        papr = 10 * np.log10(peak / (mean_power + 1e-9))
        
        # Kurtosis
        kurt = kurtosis(np.abs(complex_data), axis=1)
        
        return np.column_stack((mean_power, variance, papr, kurt))

    def close(self):
        self.f.close()

In [ ]:
processor = MachineA_Preprocessor()
processor.process_and_save()
processor.close()
print("\nDone!")


--- Processing TRAIN Set ---
sampling 1000 from 1789132 available samples.
Loading Clean I/Q Data...
Extracting Features for CLEAN signals...
Generating Barrage Noise & Extracting Features...
Saved: machine_a_X_train.npy (Shape: (2000, 4))

--- Processing VAL Set ---
sampling 1000 from 383385 available samples.
Loading Clean I/Q Data...
Extracting Features for CLEAN signals...
Generating Barrage Noise & Extracting Features...
Saved: machine_a_X_val.npy (Shape: (2000, 4))

Machine A Data is ready for training!
